# STELLAR: Spectral-Temporal Ensemble Learning with Latent Adaptive Representations
**Notebook 2 of 2** — STELLAR model training, ablation study, and hyperparameter sensitivity.

Five key innovations:
1. **Koopman Latent Dynamics** — stable linear propagation in lifted latent space
2. **Adaptive Spectral Filtering** — per-sample hypernetwork-modulated FFT filter
3. **Multi-Resolution Temporal Patching** — cross-scale attention over patches at scales {4,8,16}
4. **Neural-Process Probabilistic Head** — calibrated predictive intervals via ELBO
5. **Gated Prediction Mixture** — learned softmax routing across all prediction paths


In [ ]:
# ── 0. Update DATA_ROOT ───────────────────────────────────────────────────────
DATA_ROOT = '/kaggle/input/datasets/rahuldray12324/six-datasets/all_six_datasets'
# DATA_ROOT = '/path/to/all_six_datasets'   # ← change for local runs


In [ ]:
# ── 1. Imports & Config ───────────────────────────────────────────────────────
import os, sys, time, math, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import autocast, GradScaler
from sklearn.preprocessing import StandardScaler
from concurrent.futures import ThreadPoolExecutor, as_completed

warnings.filterwarnings('ignore')
torch.backends.cudnn.benchmark = True

SEQ_LEN   = 96
PRED_LEN  = 24
BATCH     = 64
LR        = 1e-3
EPOCHS    = 20
PATIENCE  = 5
KL_WEIGHT = 1e-3
N_GPUS    = max(1, torch.cuda.device_count())

DATA_PATHS = {k: f'{DATA_ROOT}/{k}/Y_df.csv'
              for k in ['ETTh1','ETTh2','ETTm1','ETTm2','Weather','ECL']}
print(f'GPUs: {N_GPUS}')


In [ ]:
# ── 2. Dataset ────────────────────────────────────────────────────────────────
class TSDataset(Dataset):
    def __init__(self, d, sl, pl):
        self.x=torch.tensor(d,dtype=torch.float32); self.sl=sl; self.pl=pl
    def __len__(self): return len(self.x)-self.sl-self.pl+1
    def __getitem__(self,i): return self.x[i:i+self.sl], self.x[i+self.sl:i+self.sl+self.pl]

def load_dataset(name):
    df  = pd.read_csv(DATA_PATHS[name])
    col = 'y' if 'y' in df.columns else next(c for c in df.columns if pd.api.types.is_numeric_dtype(df[c]))
    vals = df[col].values.astype(np.float32)
    n = len(vals); t1,t2 = int(.7*n), int(.8*n)
    sc = StandardScaler()
    tr = sc.fit_transform(vals[:t1].reshape(-1,1)).flatten()
    va = sc.transform(vals[t1:t2].reshape(-1,1)).flatten()
    te = sc.transform(vals[t2:].reshape(-1,1)).flatten()
    mad = float(np.mean(np.abs(np.diff(tr))))
    def dl(d,sh): return DataLoader(TSDataset(d,SEQ_LEN,PRED_LEN),BATCH,shuffle=sh,num_workers=2,pin_memory=True)
    return dl(tr,True), dl(va,False), dl(te,False), sc, mad


In [ ]:
# ── 3. STELLAR Components ─────────────────────────────────────────────────────

class RevIN(nn.Module):
    def __init__(self, eps=1e-5):
        super().__init__()
        self.eps=eps; self.w=nn.Parameter(torch.ones(1)); self.b=nn.Parameter(torch.zeros(1))
        self._mu=None; self._sig=None
    def norm(self,x):
        self._mu=x.mean(-1,keepdim=True); self._sig=x.std(-1,keepdim=True).clamp(min=self.eps)
        return (x-self._mu)/self._sig*self.w+self.b
    def denorm(self,x): return (x-self.b)/(self.w+self.eps)*self._sig+self._mu

class AdaptiveSpectralFilter(nn.Module):
    def __init__(self, sl=SEQ_LEN, d=64):
        super().__init__()
        self.sl=sl; nf=sl//2+1; self.nf=nf
        self.br=nn.Parameter(torch.ones(nf)); self.bi_=nn.Parameter(torch.zeros(nf))
        self.hyper=nn.Sequential(nn.Linear(sl,d),nn.GELU(),nn.Linear(d,nf*2))
    def forward(self,x):
        d=self.hyper(x); dr,di=d[:,:self.nf],d[:,self.nf:]
        fr,fi=self.br+dr,self.bi_+di
        xf=torch.fft.rfft(x,dim=-1)
        yr=xf.real*fr-xf.imag*fi; yi=xf.real*fi+xf.imag*fr
        return torch.fft.irfft(torch.complex(yr,yi),n=self.sl,dim=-1)

class KoopmanDynamics(nn.Module):
    def __init__(self, sl=SEQ_LEN, pl=PRED_LEN, d=64, nm=32):
        super().__init__()
        self.pl=pl; self.d=d
        self.enc=nn.Sequential(nn.Linear(sl,d*2),nn.LayerNorm(d*2),nn.GELU(),
                               nn.Dropout(.1),nn.Linear(d*2,d),nn.LayerNorm(d))
        self.U=nn.Parameter(torch.randn(d,nm)*.02); self.V=nn.Parameter(torch.randn(d,nm)*.02)
        self._lr=nn.Parameter(torch.full((nm,),.5)); self.theta=nn.Parameter(torch.randn(nm)*.1)
        self.dec=nn.Sequential(nn.Linear(d*pl,d*2),nn.GELU(),nn.Linear(d*2,pl))
    def _K(self):
        r=-F.softplus(self._lr); lr=torch.exp(r)*torch.cos(self.theta); li=torch.exp(r)*torch.sin(self.theta)
        return self.U@torch.diag(lr)@self.V.T, self.U@torch.diag(li)@self.V.T
    def forward(self,x):
        B=x.size(0); zr=self.enc(x); zi=torch.zeros_like(zr); Kr,Ki=self._K(); st=[]
        for _ in range(self.pl):
            nr=zr@Kr.T-zi@Ki.T; ni=zr@Ki.T+zi@Kr.T; zr,zi=nr,ni; st.append(zr)
        return self.dec(torch.stack(st,1).reshape(B,-1))

class MRTP(nn.Module):
    def __init__(self, sl=SEQ_LEN, d=64):
        super().__init__()
        self.sc=[4,8,16]; self.np_=[sl//s for s in self.sc]
        self.emb=nn.ModuleList([nn.Linear(s,d) for s in self.sc])
        self.pos=nn.ParameterList([nn.Parameter(torch.randn(1,n,d)*.02) for n in self.np_])
        el=lambda:nn.TransformerEncoderLayer(d,4,d*2,.1,batch_first=True,norm_first=True)
        self.tf=nn.ModuleList([nn.TransformerEncoder(el(),2) for _ in self.sc])
        self.ca=nn.MultiheadAttention(d,4,batch_first=True,dropout=.1)
        self.ln=nn.LayerNorm(d); self.pool=nn.AdaptiveAvgPool1d(1); self.proj=nn.Linear(d,d)
    def forward(self,x):
        B=x.size(0); toks=[]
        for s,n,emb,pos,tf in zip(self.sc,self.np_,self.emb,self.pos,self.tf):
            p=x[:,:n*s].reshape(B,n,s); toks.append(tf(emb(p)+pos))
        tok=torch.cat(toks,1); fused,_=self.ca(tok,tok,tok); fused=self.ln(tok+fused)
        return self.proj(self.pool(fused.transpose(1,2)).squeeze(-1))

class NPHead(nn.Module):
    def __init__(self, dc, pl, dl=32, dh=128):
        super().__init__()
        self.pl=pl; self.d_lat=dl
        self.qmu=nn.Sequential(nn.Linear(dc,dh),nn.GELU(),nn.Linear(dh,dl))
        self.qlv=nn.Sequential(nn.Linear(dc,dh),nn.GELU(),nn.Linear(dh,dl))
        self.dec=nn.Sequential(nn.Linear(dc+dl,dh),nn.GELU(),nn.Linear(dh,dh),nn.GELU())
        self.muh=nn.Linear(dh,pl); self.lvh=nn.Linear(dh,pl)
    def forward(self,ctx):
        mu_z=self.qmu(ctx); lv_z=self.qlv(ctx)
        z=mu_z+(torch.randn_like(mu_z)*torch.exp(.5*lv_z) if self.training else 0)
        h=self.dec(torch.cat([ctx,z],-1))
        kl=-0.5*(1+lv_z-mu_z.pow(2)-lv_z.exp()).sum(-1).mean()
        return self.muh(h), F.softplus(self.lvh(h))+1e-6, kl

print('All STELLAR components defined ✓')


In [ ]:
# ── 4. Full STELLAR Model ─────────────────────────────────────────────────────
class STELLAR(nn.Module):
    def __init__(self, sl=SEQ_LEN, pl=PRED_LEN, d=64, dk=64, nm=32, dl=32):
        super().__init__()
        self.pl=pl
        self.revin=RevIN(); self.asf=AdaptiveSpectralFilter(sl); self.asp=nn.Linear(sl,d)
        self.koop=KoopmanDynamics(sl,pl,dk,nm); self.mrtp=MRTP(sl,d)
        dc=d*2
        self.linear=nn.Linear(sl,pl,bias=False)
        self.nphead=NPHead(dc,pl,dl)
        self.gate=nn.Sequential(nn.Linear(dc,3),nn.Softmax(dim=-1))
    def forward(self,x):
        xn=self.revin.norm(x)
        fsp=self.asp(self.asf(xn)); koop=self.koop(xn)
        fmr=self.mrtp(xn); lin=self.linear(xn)
        ctx=torch.cat([fsp,fmr],-1); g=self.gate(ctx)
        gk,gl,gn=g[:,0:1],g[:,1:2],g[:,2:3]
        npm,npv,kl=self.nphead(ctx)
        mean=self.revin.denorm(gk*koop+gl*lin+gn*npm)
        var=npv*self.revin._sig**2
        return mean,var,kl

# Quick sanity check
_m = STELLAR()
_x = torch.randn(4, SEQ_LEN)
_mu,_var,_kl = _m(_x)
print(f'STELLAR sanity check ✓  mean={_mu.shape}  var={_var.shape}  kl={_kl.item():.4f}')
print(f'Params: {sum(p.numel() for p in _m.parameters() if p.requires_grad):,}')


In [ ]:
# ── 5. Losses & Metrics ───────────────────────────────────────────────────────
def gnll(mu,var,y): return (0.5*torch.log(var)+0.5*(y-mu).pow(2)/var).mean()
def crps_g(mu,sig,y):
    n=torch.distributions.Normal(0.,1.); z=(y-mu)/sig.clamp(1e-8)
    return (sig*(z*(2*n.cdf(z)-1)+2*n.log_prob(z).exp()-1/math.sqrt(math.pi))).mean().item()
def compute_metrics(pred,var,tgt,mad):
    mae=(pred-tgt).abs().mean().item(); rmse=((pred-tgt)**2).mean().sqrt().item()
    sp=(2*(pred-tgt).abs()/(pred.abs()+tgt.abs()+1e-8)).mean().item()*100
    mase=mae/(mad+1e-8); std=var.clamp(1e-10).sqrt()
    return mae,rmse,sp,mase,crps_g(pred,std,tgt),gnll(pred,var.clamp(1e-10),tgt).item()


In [ ]:
# ── 6. Training Engine ────────────────────────────────────────────────────────
def train_stellar(name, gpu_id, model_factory=STELLAR):
    device=torch.device(f'cuda:{gpu_id}' if torch.cuda.is_available() else 'cpu')
    print(f'[{name}] → GPU {gpu_id}'); t0=time.time()
    tr_dl,va_dl,te_dl,sc,mad=load_dataset(name)
    model=model_factory().to(device)
    npar=sum(p.numel() for p in model.parameters() if p.requires_grad)
    opt=torch.optim.AdamW(model.parameters(),lr=LR,weight_decay=1e-4)
    sched=torch.optim.lr_scheduler.CosineAnnealingLR(opt,T_max=EPOCHS,eta_min=1e-5)
    scaler=GradScaler(); best_v,best_st,pat=float('inf'),None,0
    for ep in range(1,EPOCHS+1):
        model.train(); tl=0.
        for x,y in tr_dl:
            x,y=x.to(device),y.to(device); opt.zero_grad(set_to_none=True)
            with autocast():
                mu,var,kl=model(x); loss=gnll(mu,var,y)+KL_WEIGHT*kl
            scaler.scale(loss).backward(); scaler.unscale_(opt)
            nn.utils.clip_grad_norm_(model.parameters(),1.); scaler.step(opt); scaler.update()
            tl+=loss.item()
        sched.step()
        model.eval(); vl=0.
        with torch.no_grad():
            for x,y in va_dl:
                x,y=x.to(device),y.to(device)
                with autocast():
                    mu,var,kl=model(x); vl+=gnll(mu,var,y).item()
        vl/=max(len(va_dl),1)
        print(f'  [{name}] ep{ep:02d}  tr={tl/len(tr_dl):.4f}  val={vl:.4f}  lr={sched.get_last_lr()[0]:.2e}')
        if vl<best_v-1e-6: best_v=vl; pat=0; best_st={k:v.cpu().clone() for k,v in model.state_dict().items()}
        else:
            pat+=1
            if pat>=PATIENCE: print(f'  [{name}] early stop @ ep {ep}'); break
    if best_st: model.load_state_dict({k:v.to(device) for k,v in best_st.items()})
    model.eval(); amu,avar,ay=[],[],[]
    with torch.no_grad():
        for x,y in te_dl:
            mu,var,_=model(x.to(device)); amu.append(mu.cpu()); avar.append(var.cpu()); ay.append(y)
    mu_t=torch.cat(amu); var_t=torch.cat(avar).clamp(1e-10); y_t=torch.cat(ay)
    mae,rmse,smape,mase,crps,nll=compute_metrics(mu_t,var_t,y_t,mad)
    elapsed=round(time.time()-t0,1)
    print(f'\n  ✓ [{name}]  MAE={mae:.4f}  RMSE={rmse:.4f}  sMAPE={smape:.2f}%  '
          f'MASE={mase:.4f}  CRPS={crps:.4f}  NLL={nll:.4f}  params={npar:,}  time={elapsed}s\n')
    return dict(model=model.__class__.__name__, dataset=name,
                mae=round(mae,4), rmse=round(rmse,4), smape=round(smape,4),
                mase=round(mase,4), crps=round(crps,4), nll=round(nll,4),
                n_params=npar, train_time_sec=elapsed)


In [ ]:
# ── 7. Run STELLAR on all datasets ────────────────────────────────────────────
DATASETS = ['ETTh1','ETTh2','ETTm1','ETTm2','Weather','ECL']
BEST_BASE = {'ECL':0.2512,'ETTh1':0.4685,'ETTh2':0.3210,'ETTm1':0.3794,'ETTm2':0.2564,'Weather':0.0468}

results = []
gpu_map = {ds: i % N_GPUS for i,ds in enumerate(DATASETS)}

print('='*60); print(' STELLAR — starting parallel training'); print('='*60)
with ThreadPoolExecutor(max_workers=N_GPUS) as ex:
    futures = {ex.submit(train_stellar, ds, gpu_map[ds]): ds for ds in DATASETS}
    for fut in as_completed(futures):
        try: results.append(fut.result())
        except Exception as e: print(f'[ERROR] {futures[fut]}: {e}')

df_stellar = pd.DataFrame(results).sort_values('dataset').reset_index(drop=True)
df_stellar.to_csv('stellar_results.csv', index=False)
print('\n'+'='*60); print(' FINAL RESULTS'); print('='*60)
print(df_stellar.to_string(index=False))

print('\n── MAE improvement vs best baseline ──')
for _,row in df_stellar.iterrows():
    base=BEST_BASE.get(row['dataset']); delta=(base-row['mae'])/base*100 if base else 0
    print(f"  {'✓' if delta>0 else '✗'} {row['dataset']:8s}  STELLAR={row['mae']:.4f}  base={base:.4f}  Δ={delta:+.1f}%")


## Ablation Study

Each ablation variant removes exactly one component of STELLAR, allowing us to measure each component's individual contribution.


In [ ]:
# ── 8. Ablation Variants ──────────────────────────────────────────────────────
class STELLAR_NoKoopman(STELLAR):
    def forward(self,x):
        xn=self.revin.norm(x); fsp=self.asp(self.asf(xn))
        koop=torch.zeros(xn.size(0),self.pl,device=x.device)
        fmr=self.mrtp(xn); lin=self.linear(xn); ctx=torch.cat([fsp,fmr],-1)
        g=self.gate(ctx); npm,npv,kl=self.nphead(ctx)
        mean=self.revin.denorm(g[:,0:1]*koop+g[:,1:2]*lin+g[:,2:3]*npm)
        return mean,npv*self.revin._sig**2,kl

class STELLAR_NoASF(STELLAR):
    def forward(self,x):
        xn=self.revin.norm(x); fsp=self.asp(xn)  # bypass ASF
        koop=self.koop(xn); fmr=self.mrtp(xn); lin=self.linear(xn)
        ctx=torch.cat([fsp,fmr],-1); g=self.gate(ctx); npm,npv,kl=self.nphead(ctx)
        mean=self.revin.denorm(g[:,0:1]*koop+g[:,1:2]*lin+g[:,2:3]*npm)
        return mean,npv*self.revin._sig**2,kl

class STELLAR_NoMRTP(STELLAR):
    def __init__(self,*a,**kw):
        super().__init__(*a,**kw)
        self.simple=nn.Linear(SEQ_LEN, self.mrtp.proj.out_features)
    def forward(self,x):
        xn=self.revin.norm(x); fsp=self.asp(self.asf(xn))
        koop=self.koop(xn); fmr=self.simple(xn); lin=self.linear(xn)
        ctx=torch.cat([fsp,fmr],-1); g=self.gate(ctx); npm,npv,kl=self.nphead(ctx)
        mean=self.revin.denorm(g[:,0:1]*koop+g[:,1:2]*lin+g[:,2:3]*npm)
        return mean,npv*self.revin._sig**2,kl

class STELLAR_NoGating(STELLAR):
    def forward(self,x):
        xn=self.revin.norm(x); fsp=self.asp(self.asf(xn))
        koop=self.koop(xn); fmr=self.mrtp(xn); lin=self.linear(xn)
        ctx=torch.cat([fsp,fmr],-1); npm,npv,kl=self.nphead(ctx)
        B=xn.size(0); w=torch.full((B,1),1/3,device=x.device)
        mean=self.revin.denorm(w*koop+w*lin+w*npm)
        return mean,npv*self.revin._sig**2,kl

class STELLAR_NoNPHead(STELLAR):
    def __init__(self,*a,**kw):
        super().__init__(*a,**kw)
        self.det_var=nn.Linear(64*2, PRED_LEN)  # d_ctx = d*2 = 128
    def forward(self,x):
        xn=self.revin.norm(x); fsp=self.asp(self.asf(xn))
        koop=self.koop(xn); fmr=self.mrtp(xn); lin=self.linear(xn)
        ctx=torch.cat([fsp,fmr],-1)
        z=torch.zeros(ctx.size(0),self.nphead.d_lat,device=x.device)
        h=self.nphead.dec(torch.cat([ctx,z],-1)); npm=self.nphead.muh(h)
        npv=F.softplus(self.det_var(ctx))+1e-6; kl=torch.tensor(0.,device=x.device)
        g=self.gate(ctx); mean=self.revin.denorm(g[:,0:1]*koop+g[:,1:2]*lin+g[:,2:3]*npm)
        return mean,npv*self.revin._sig**2,kl

ABLATION_VARIANTS = {
    'STELLAR':          STELLAR,
    'NoKoopman':        STELLAR_NoKoopman,
    'NoAdaptiveFilter': STELLAR_NoASF,
    'NoMRTP':           STELLAR_NoMRTP,
    'NoGating':         STELLAR_NoGating,
    'NoNPHead':         STELLAR_NoNPHead,
}
print('Ablation variants defined ✓')


In [ ]:
# ── 9. Run Ablations ──────────────────────────────────────────────────────────
# Note: runs all 6 datasets × 6 variants. Takes significant time on 2 GPUs.
# For a quick check, set ABL_DATASETS = ['ETTh1', 'ETTh2']

ABL_DATASETS = DATASETS  # or reduce for faster testing

abl_results = []
for vname, VCls in ABLATION_VARIANTS.items():
    print(f'\n{"="*55}\nAblation: {vname}\n{"="*55}')
    gpu_map = {ds: i % N_GPUS for i,ds in enumerate(ABL_DATASETS)}
    with ThreadPoolExecutor(max_workers=N_GPUS) as ex:
        futures = {ex.submit(train_stellar, ds, gpu_map[ds], VCls): ds for ds in ABL_DATASETS}
        for fut in as_completed(futures):
            try:
                r=fut.result(); r['variant']=vname; abl_results.append(r)
            except Exception as e: print(f'  [ERROR] {vname}/{futures[fut]}: {e}')

df_abl = pd.DataFrame(abl_results).sort_values(['variant','dataset'])
df_abl.to_csv('ablation_results.csv', index=False)

pivot = df_abl.pivot_table(index='dataset', columns='variant', values='mae').round(4)
print('\nAblation MAE (lower is better):')
pivot


## Hyperparameter Sensitivity

Varying the number of Koopman spectral modes on Weather and ECL.


In [ ]:
# ── 10. Hyperparameter Sensitivity ───────────────────────────────────────────
HP_DATASETS = ['Weather', 'ECL']
N_MODES_LIST = [16, 32, 64, 128]

hp_results = []
for ds in HP_DATASETS:
    print(f'\nSweep on {ds}')
    for nm in N_MODES_LIST:
        print(f'  n_modes={nm}')
        r = train_stellar(ds, 0, lambda m=nm: STELLAR(nm=m))
        r['n_modes'] = nm; hp_results.append(r)

df_hp = pd.DataFrame(hp_results).sort_values(['dataset','n_modes'])
df_hp.to_csv('hyperparam_sensitivity.csv', index=False)
print('\nHyperparameter sensitivity:')
df_hp[['dataset','n_modes','mae','rmse','crps']].to_string(index=False)


In [ ]:
# ── 11. Summary ───────────────────────────────────────────────────────────────
print('\n' + '='*60)
print('All results saved:')
print('  stellar_results.csv')
print('  ablation_results.csv')
print('  hyperparam_sensitivity.csv')
print('='*60)
print('\nSTELLAR final results:')
print(df_stellar[['dataset','mae','rmse','smape','mase','crps','nll']].to_string(index=False))
